In [1]:
import os
from dotenv import load_dotenv

# Load environment variables (.env)
load_dotenv()

# Enable LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_API_KEY"] = "your_langchain_api_key_here" # Ensure this is in your .env
os.environ["LANGCHAIN_PROJECT"] = "OmniQuery_Step3_Prototype"

# Database Configuration
DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "enterprise_hub"),
    "user": os.getenv("DB_USER", "admin"),
    "password": os.getenv("DB_PASSWORD", "secretpassword"),
}

MAX_RETRIES = 3

agent state 

In [2]:
from typing import TypedDict, Annotated, List, Dict, Any, Optional
import operator

class AgentState(TypedDict):
    messages: Annotated[List[Dict[str, Any]], operator.add]
    sql_query: Optional[str]
    sql_result: Optional[List[Dict[str, Any]]]
    sql_error: Optional[str]
    sql_retry_count: int
    sql_validation_error: Optional[str]
    execution_status: str  # "success", "retry", "failed", "blocked"
    final_response: Optional[str] # ADDED: The human-readable answer

Node 1 — SQL Generator

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

def get_database_schema() -> str:
    return """
    Table: product_sales
    Columns:
      - id (SERIAL PRIMARY KEY)
      - region (VARCHAR(50)) -- 'North America', 'Europe', 'Asia-Pacific'
      - product_line (VARCHAR(100)) -- 'AR Interior Designer Pro (License)', 'Computer Vision API Tracker'
      - revenue (NUMERIC(12, 2))
      - units_sold (INT)
      - fiscal_quarter (VARCHAR(10)) -- 'Q1-2026'
    """

def sql_generator_node(state: AgentState) -> Dict[str, Any]:
    user_query = state["messages"][-1]["content"]
    last_error = state.get("sql_error")
    retry_count = state.get("sql_retry_count", 0)
    schema = get_database_schema()

    system_prompt = f"""You are the SQL generation engine for OmniQuery, an enterprise analytics system.
Your job is to convert the user's analytical request into a PostgreSQL SELECT query.

Database schema:
{schema}

Rules:
1. Generate PostgreSQL-compatible SQL.
2. Use ONLY tables and columns in the provided schema.
3. Only generate read-only queries.
4. Never use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, or CREATE.
5. Return ONLY the raw SQL. Do NOT use markdown formatting (no ```sql).
6. Do not invent columns.
7. If the user asks for data outside the schema, write a query that returns 0 rows safely (e.g., SELECT 0 WHERE 1=0).
"""

    if last_error:
        # Retry Prompt
        prompt = system_prompt + f"""
IMPORTANT - PREVIOUS QUERY FAILED!
Previous SQL: {state.get('sql_query')}
PostgreSQL Error: {last_error}

Fix the SQL query to resolve the error while preserving the original intent:
User Request: {user_query}
"""
    else:
        # Initial Prompt
        prompt = system_prompt + f"\nUser Request: {user_query}"

    response = llm.invoke(prompt)
    
    # Clean LLM output to guarantee valid SQL execution
    clean_sql = response.content.strip().replace("```sql", "").replace("```", "").strip()

    return {
        "sql_query": clean_sql,
        "sql_retry_count": retry_count + (1 if last_error else 0),
        "sql_validation_error": None, # Reset validation on new generation
        "sql_error": None # Reset execution error on new generation
    }

d:\rag-projects\OmniQuery--multiagent-sql-rag-analyst\omniquery-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Node 2 — SQL Validator

In [4]:
import re

FORBIDDEN_KEYWORDS = [
    "INSERT", "UPDATE", "DELETE", "DROP", "ALTER", 
    "TRUNCATE", "CREATE", "GRANT", "REVOKE", "COPY", "CALL", "DO"
]

def sql_validator_node(state: AgentState) -> Dict[str, Any]:
    sql = state.get("sql_query", "").upper()
    
    # 1. Check for forbidden keywords using regex word boundaries
    for keyword in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{keyword}\b", sql):
            return {
                "sql_validation_error": f"Forbidden keyword detected: {keyword}",
                "execution_status": "blocked"
            }
            
    # 2. Check for multiple statements (semicolon injection)
    # A single semicolon at the end is fine, but multiple are dangerous.
    if sql.count(";") > 1 or (sql.count(";") == 1 and not sql.strip().endswith(";")):
        return {
            "sql_validation_error": "Multiple SQL statements detected.",
            "execution_status": "blocked"
        }

    return {
        "sql_validation_error": None,
        "execution_status": "safe" # Proceed to execution
    }

Node 3 — SQL Executor

In [5]:
import psycopg2
from psycopg2.extras import RealDictCursor

def sql_executor_node(state: AgentState) -> Dict[str, Any]:
    sql = state["sql_query"]
    
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        # Set read-only transaction mode as an extra layer of database safety
        conn.set_session(readonly=True) 
        cursor = conn.cursor(cursor_factory=RealDictCursor)
        cursor.execute(sql)
        results = cursor.fetchall()
        conn.close()

        return {
            "sql_result": [dict(row) for row in results],
            "sql_error": None,
            "execution_status": "success"
        }

    except Exception as e:
        error_msg = str(e).strip()
        return {
            "sql_result": None,
            "sql_error": error_msg,
            "execution_status": "retry"
        }

Graph Routers

In [6]:
def route_after_validator(state: AgentState) -> str:
    """Routes to Executor if SAFE, otherwise ends graph if BLOCKED."""
    if state.get("execution_status") == "blocked":
        return "blocked"
    return "safe"

def route_after_executor(state: AgentState) -> str:
    """Routes to SUCCESS, FAILED (Max retries hit), or RETRY."""
    status = state.get("execution_status")
    
    if status == "success":
        return "success"
    
    if status == "retry":
        if state.get("sql_retry_count", 0) >= MAX_RETRIES:
            return "failed"
        return "retry"
        
    return "failed"

NODE 4 - Responder node

In [7]:
def sql_responder_node(state: AgentState) -> dict:
    """Node: Converts raw JSON results or errors into a natural language response."""
    status = state.get("execution_status")
    user_query = state["messages"][-1]["content"]
    
    # 1. Handle Blocked / Unsafe Queries
    if status == "blocked":
        return {"final_response": f"I cannot execute this request: {state.get('sql_validation_error')}"}
        
    # 2. Handle Exhausted Retries
    if status == "failed":
        return {"final_response": "I'm sorry, I couldn't retrieve the data due to a database error."}
        
    # 3. Handle Empty / Null Results
    results = state.get("sql_result")
    if not results:
        return {"final_response": "I checked the database, but no matching records were found or the data does not exist."}
        
    # 4. Format Successful Data into Natural Language
    prompt = f"""
    You are a helpful enterprise data analyst. Answer the user's question using ONLY the provided data.
    Keep the answer concise, professional, and natural. Do not mention "SQL", "database", or "JSON".
    Format currency appropriately.

    User Request: {user_query}
    Raw Data: {results}
    """
    
    response = llm.invoke(prompt)
    return {"final_response": response.content.strip()}

Assemble & Compile LangGraph

In [8]:
from langgraph.graph import StateGraph, END

# Build and export graph
builder = StateGraph(AgentState)
builder.add_node("generator", sql_generator_node)
builder.add_node("validator", sql_validator_node)
builder.add_node("executor", sql_executor_node)
builder.add_node("responder", sql_responder_node) # ADDED

builder.set_entry_point("generator")
builder.add_edge("generator", "validator")

# Route to Executor if safe, or skip straight to Responder if blocked
builder.add_conditional_edges(
    "validator", 
    route_after_validator, 
    {"safe": "executor", "blocked": "responder"}
)

# Route to Responder on success/fail, or loop back to Generator on retry
builder.add_conditional_edges(
    "executor", 
    route_after_executor, 
    {"success": "responder", "retry": "generator", "failed": "responder"}
)

# Responder is the final step
builder.add_edge("responder", END)

sql_agent_graph = builder.compile()
print(sql_agent_graph.get_graph().draw_ascii())

            +-----------+             
            | __start__ |             
            +-----------+             
                  *                   
                  *                   
                  *                   
            +-----------+             
            | generator |             
            +-----------+             
           ***         ...            
          *               .           
        **                 ...        
+-----------+                 .       
| validator |                 .       
+-----------+...              .       
      .         .....         .       
      .              ...      .       
      .                 ...   .       
      ..                +----------+  
        .               | executor |  
         ...            +----------+  
            .          ...            
             ...      .               
                .   ..                
            +-----------+             
            | responder |

TEST 

In [10]:
def run_test_matrix():
    tests = [
        {"name": "Test 1 — Simple aggregation", "query": "What is the total revenue in North America?"},
        {"name": "Test 2 — Filtering", "query": "How many units of AR Interior Designer Pro were sold in Europe?"},
        {"name": "Test 3 — Grouping", "query": "Show total revenue grouped by region."},
        {"name": "Test 4 — Multiple conditions", "query": "What was the revenue for the Computer Vision API Tracker in Asia-Pacific?"},
        {"name": "Test 5 — Intentional SQL error", "query": "Select the sum of revenues (use exactly the word 'revenues' as the column name) from product_sales."},
        {"name": "Test 6 — Unknown request", "query": "What was Apple's revenue in 2025?"},
        {"name": "Test 7 — Dangerous request", "query": "Delete all sales data from the database."},
        {"name": "Test 8 — Retry exhaustion", "query": "Write a query that intentionally throws a syntax error and cannot be fixed, like SELECT * FROM non_existent_table_999."}
    ]

    for i, test in enumerate(tests, 1):
        print(f"\n{'='*60}")
        print(f"▶️ {test['name']}")
        print(f"User Query:   {test['query']}")
        print(f"{'-'*60}")
        
        initial_state = {
            "messages": [{"role": "user", "content": test["query"]}],
            "sql_retry_count": 0,
            "sql_error": None
        }

        # Invoke the graph
        final_state = sql_agent_graph.invoke(initial_state)

        # Print Execution Overview
        print(f"Status:       {final_state.get('execution_status', '').upper()}")
        print(f"Retries:      {final_state.get('sql_retry_count')}")
        print(f"Final SQL:    {final_state.get('sql_query')}")
        
        # Print Debug Technical Details (Optional)
        if final_state.get("sql_result") is not None:
            print(f"Raw Data:     {final_state.get('sql_result')}")
        elif final_state.get("sql_validation_error"):
            print(f"Validation:   {final_state.get('sql_validation_error')}")
        elif final_state.get("sql_error"):
            print(f"Error Trace:  {final_state.get('sql_error')}")
            
        # Print Final Human-Readable Output
        print(f"{'-'*60}")
        print(f"💬 Answer:    {final_state.get('final_response')}")

if __name__ == "__main__":
    run_test_matrix()


▶️ Test 1 — Simple aggregation
User Query:   What is the total revenue in North America?
------------------------------------------------------------
Status:       SUCCESS
Retries:      0
Final SQL:    SELECT SUM(revenue) FROM product_sales WHERE region = 'North America'
Raw Data:     [{'sum': Decimal('2100000.00')}]
------------------------------------------------------------
💬 Answer:    The total revenue in North America is $2,100,000.00.

▶️ Test 2 — Filtering
User Query:   How many units of AR Interior Designer Pro were sold in Europe?
------------------------------------------------------------
Status:       SUCCESS
Retries:      0
Final SQL:    SELECT SUM(units_sold) FROM product_sales WHERE product_line = 'AR Interior Designer Pro (License)' AND region = 'Europe'
Raw Data:     [{'sum': 600}]
------------------------------------------------------------
💬 Answer:    According to the data, 600 units of AR Interior Designer Pro were sold in Europe.

▶️ Test 3 — Grouping
User Query